# Trend analysis observations Notebook
## Probabilistic method

---

## How to Use This Notebook

**1. Follow the numbered steps in order.**  
Each section builds upon the previous one, from setup, data loading, and climatology computation, to event analysis and visualization.

**2. Look for <font color="orange"> Orange cells  </font> and code cells marked as <font color="lightgreen">##### (User selection) ##### </font>:** 
| <font color="orange"> Orange cells  </font> | <font color="orange"> Need user intervantion </font>|
| ----------- | ----------- |
| <font color="green">**Green cells** </font> | <font color="green">**Run automatically on user input provided in the orange cells and should not be adjusted in most cases** </font>|


**3. Run cells sequentially.**  
Start from the top and execute each cell (`Shift` + `Enter`).  

### <font color="green"> Import require packages </font>

In [ ]:
from datetime import datetime, timedelta
from c3s_event_attribution_tools import *
import xarray as xr
import pandas as pd
import geopandas as gpd
import os
import rpy2.robjects as ro
from rpy2.robjects.packages import importr

# import R libraries from WWA
ro.r('''
if (!requireNamespace("remotes", quietly = TRUE)) {
    install.packages("remotes", repos="https://cloud.r-project.org", quiet=TRUE)
}

# Force install the older, compatible version of gsl
if (!requireNamespace("gsl", quietly = TRUE)) {
    remotes::install_version("gsl", version = "2.1-8", repos = "https://cloud.r-project.org")
}

# Now install rwwa
remotes::install_github("WorldWeatherAttribution/rwwa")
''')

rwwa = importr("rwwa")
%load_ext rpy2.ipython

### <font color="orange"> User specifications </font>

In [ ]:
# Directory you wish to store output files in. using ../ specifies the parent directory
CURRENT_DIRECTORY = os.getcwd() # do not touch, __file__ specifies the current directory of the file

################# (User selection) ###################
your_save_directory = os.path.abspath(os.path.join(CURRENT_DIRECTORY, "./data"))   # change ../data to your desired directory
######################################################

### <font color="orange"> Choice of parameter </font>

In [ ]:
# Choice of parameter (Tmax, Tmean, Tmin, Precipitation)
################# (User selection) ###################
parameter = "" 
event_end = datetime(, , ) #YYYY,MM,DD 
bbox = (, , , ) # (Southern boundary, Western boundary, Northern boundary, Eastern boundary)
your_api_key = ''

######################################################
event_start = event_end - timedelta(days=14)
bbox = Utils.convert_bbox(*bbox)
event_year = event_end.year
######################################################

# Get parameter configuration
config = Utils.get_parameter_config(parameter)
value_col = config["value_col"]
y_label = config["y_label"]
unit = config["unit"]
calculation = config["calculation"] 
if parameter in ["Tmax", "Tmean", "Tmin"]:
    variable = "Temperature"
    fit_type = "shift"
    method = "std"
elif parameter == "Precipitation":
    variable = "Precipitation"
    fit_type = "fixeddisp"
    method = "dispersion"

## 3.2 Check (visually) for inhomogeneity of annual event time series

### <font color='orange'>Please specify the file name of the annual time series from the event definition step</font>

In [ ]:
################# (User selection) ###################
annual_timeseries_load = 'ts_ann_studyregion.nc'
######################################################

ts_ann_studyregion = xr.open_dataset(os.path.join(your_save_directory, annual_timeseries_load)).to_dataframe().reset_index()
ts_ann_studyregion

- a. Pay special attention to years that might coincide with key events such as the introduction of satellites
    - i. 1979 (start of satellite era)
    - ii. Tropical precipitation: 1998 (start of better tropical satellite data)
- b. Jumps in the (annual) time series
- c. visually check the scale or dispersion parameter (running standard deviation)

In [ ]:
ts_ann_studyregion_15y = Process.calculate_rolling_window(gdf=ts_ann_studyregion, value_col=value_col, datetime_col="year",
                                  window=15, method=method, min_periods=1, centering=True, ci=0.95)

ts_ann_studyregion_15y

In [ ]:
Plot.plot_timeserie(data=ts_ann_studyregion_15y, datetime_col="year", value_col=value_col,
               title=f"15 year running {method} of {parameter}", x_label="Date", y_label=method, line_style='solid', ci=True);

## 3.2.d. Decide on which years to use if data is not homogeneous

### <font color='orange'>Specify the preferred time range based on the plot</font>

In [ ]:
year_range = (1950, event_year)

### <font color='green'> Slice the annual time series on the given time range </font>

In [ ]:
ts_ann_studyregion_subset = Utils.subset_gdf(gdf=ts_ann_studyregion, datetime_col="year", date_range=year_range)
ts_ann_studyregion_subset

## 3.3 Conclude about the quality of ERA5
(and potentially other datasets) for the specific event definition and implications for the study results and the years to use, write into the output table in Notes & tables and the scientific report Section 2.1.
- a. For the region, check performance of the dataset, conclude on whether the data can be used with limited problems (“satisfactory”) or caution is necessary (“caution”) and conclude on a sentence in the scientific report
    - i. Temperature, include a statement on the performance of ERA5 for temperature based on Lopes et al. (2024) Conclusions from this paper are summarized here (attached to deliverable).
    - ii. Precipitation, include a statement on the performance of ERA5 for precipitation based on Lavers et al. (2022) Conclusions from this paper are summarized here (attached to deliverable).
    - iii. Other variable: if drought, base a general statement on temperature and precipitation, and/or perform literature study on the index and region specifically.
- b. For the use of years, using plots produced in Step 3.2.b visually check whether scale (temperature) or dispersion (precipitation) parameter show a trend and only use the stationary scale/dispersion-fit if the stationarity assumption is not obviously invalid in the sense that the trend is much greater than variability. Otherwise, a non-stationary scale- or
dispersion-parameter would be more appropriate (not part of OAO protocol). Add a sentence on caution on the results should be added to the scientific report Sec. 3.1 - see also Step 3.5b.iv

## 3.4 decide on which covariate to use
(normally this will only be GMST, smoothed with a 4-year running mean to remove the El Nino signal) and make sure the 4-year smoothed ERA5 GMST is updated up to the current month.
- a. C3S defines the 1850–1900 pre‑industrial global mean surface temperature to be 0.88 °C below the 1991–2020 ERA5 global average.

In [ ]:
client = DataClient(cds_key=your_api_key, beacon_cache_url="https://beacon-era5.maris.nl/")
gmst = client.gmst_xr(time_range=(datetime(year_range[0], 1, 1), datetime.now())) # bbox entire world, end datetime today
gmst_weights = Process.weighted_values(gmst, 't2m') # xarray does not allow as to apply the weight in the same way as pandas/geopandas, therefore we only return the weight and apply it later in the calculation

gmst_monthly = (
    gmst['t2m']
    .groupby('valid_time')
    .map(lambda da: da.weighted(gmst_weights).mean(dim=('latitude', 'longitude')))
    .drop_vars(['number', 'expver'], errors='ignore')
    .to_dataframe(name='gmst')
    .reset_index()
)
gmst_monthly

In [ ]:
if gmst_monthly['valid_time'].iloc[-1].month < 4:
    gmst_monthly = gmst_monthly[gmst_monthly["valid_time"].dt.year != gmst_monthly['valid_time'].iloc[-1].year]
    gmst_yearly = gmst_monthly.groupby(gmst_monthly["valid_time"].dt.year)['gmst'].mean().reset_index()
    gmst_yearly_rolled = Process.calculate_rolling_window(gdf=gmst_yearly, value_col='gmst', datetime_col="valid_time", window=4, min_periods=2, centering=True, method="mean")
    gmst_yearly_rolled = gmst_yearly_rolled.rename(columns={"valid_time": "year"})

else:
    clim31d_xr = xr.open_dataset(os.path.join(your_save_directory, f"climatology_1991-2020_Maximum Temperature.nc"))
    clim31d = gpd.GeoDataFrame(clim31d_xr.to_dataframe().reset_index(), geometry=gpd.points_from_xy(clim31d_xr.to_dataframe().reset_index().longitude, clim31d_xr.to_dataframe().reset_index().latitude), crs="EPSG:4326")
    clim31d = clim31d.rename(columns={"dayofyear": "doy"})
    clim31d['valid_time'] = pd.to_datetime(event_end.year * 1000 + clim31d["doy"], format="%Y%j")
    studyregion = gpd.read_file(os.path.join(your_save_directory, "sf_studyregion.shp"))
    ts_clim31d_studyregion = Process.calculate_seasonal_cycle(clim31d=clim31d, studyregion=studyregion, value_col='t2m', datetime_col="valid_time", event_end=event_end)[0] 
    monthly_clim31d_studyregion = (ts_clim31d_studyregion.resample("ME", on="valid_time")['t2m'].mean().rename_axis("valid_time").reset_index())
    monthly_clim31d_studyregion["valid_time"] = monthly_clim31d_studyregion["valid_time"].dt.to_period("M").dt.to_timestamp()
    gmst_monthly_filled = Process.fill_missing_gmst_with_climatology(gmst_monthly, monthly_clim31d_studyregion, "gmst", "t2m")
    gmst_yearly = gmst_monthly_filled.groupby(gmst_monthly_filled["valid_time"].dt.year)['gmst'].mean().reset_index()
    gmst_yearly_rolled = Process.calculate_rolling_window(gdf=gmst_yearly, value_col='gmst', datetime_col="valid_time", window=4, min_periods=2, centering=True, method="mean")
    gmst_yearly_rolled = gmst_yearly_rolled.rename(columns={"valid_time": "year"})

In [ ]:
merged_gmst = pd.merge(ts_ann_studyregion_subset, gmst_yearly_rolled, left_on="year", right_on="year", how="inner") 
ref_val = merged_gmst.loc[merged_gmst["year"] == event_year, 'gmst'].values[0] # calculate anomaly relative to event year
merged_gmst_anomaly = merged_gmst.copy()
merged_gmst_anomaly["gmst"] = merged_gmst_anomaly['gmst'] - ref_val
merged_gmst_anomaly.to_csv("./data/gmst_t2m.csv", index=False)
merged_gmst_anomaly

## 3.5 Apply statistical method
information for decisions on fit properties:
- a. fit data to statistical model, and check, at least by eye, that this fit agrees with the observed data points. Show this in a figure.
    - i. Gauss - soft extremes with low return periods that are not in the tail. Threshold μ (mu), scale parameter σ (sigma). Usually used for seasonal averages.
    - ii. GEV - largest observation from a large sample: annual/seasonal maximum, block maximum. Location parameter μ, scale parameter σ, shape parameter ξ. Usually used for 1-day to 14-day averages, annual (or seasonal) maximum
    - iii. For in-depth studies potentially use Lognormal, possibly better for precipitation.
- b. shift/scale with GMST or other proxy for anthropogenic forcing
    - i. for temperature extremes the distribution shifts due to global warming without changing the shape (variability: σ, ξ constant).
    - ii. for precipitation (and wind) extremes the distribution scales (variability over mean: dispersion parameter σ/μ and ξ constant).
    - iii. For precipitation with Lognormal use shift
    - iv. Use the visual check of scale or dispersion parameter (done in Step 3.3b): use the stationary scale/dispersion-fit if the stationarity assumption is not obviously invalid. Otherwise using a non-stationary scale- or dispersion-parameter would be more appropriate, and a sentence on caution on the results should be added to the scientific report Section. 3.1
    - v. decide on which covariate to use (normally GMST, smoothed with a 4-year running mean or a similar method to remove the El Nino signal)
    - vi. Include the datapoint of the event under study in the fit

### <font color='orange'>Please specify the following variables<font>

In [ ]:
%%R -i event_year -i value_col -i fit_type
dist = "gev" #norm 
covnm = "gmst"
lower = FALSE
cooling_offset = 1.3 #fixed for now

In [ ]:
%%R -i merged_gmst_anomaly 

df <- merged_gmst_anomaly

mdl <- fit_ns(dist = dist, type = fit_type, data = df, varnm = value_col, covnm = covnm, lower = lower, ev_year = event_year)

# the factual climate should have the GMST of the year in which the event occurred
cov_factual <- data.frame(gmst = df$gmst[df$year == event_year])

# the counterfactual climate can represent any alternative climate (WWA always uses a preindustrial climate, 1.3C cooler than the present)
cov_counterfactual <- data.frame(gmst = df$gmst[df$year == event_year] - cooling_offset)

In [ ]:
bounds = np.array([merged_gmst_anomaly[value_col].min(), merged_gmst_anomaly[value_col].max()])

In [ ]:
%%R -i bounds
# what does the fitted trend look like over time?
# `add_loess = T` will add a nonparametric smoother - use this to check whether the fitted model captures the observed trend
plot_trend(mdl, add_loess = T, ylim = c(bounds[1],bounds[2]))

In [ ]:
%%R -i bounds
# what does the fitted trend look like vs GMST?
plot_covtrend(mdl, xcov = covnm, add_loess = T, ylim = c(bounds[1],bounds[2]))

In [ ]:
%%R -i bounds
# How well does the model fit the data?
# the points should be close to the line - if they're not within the shaded region, the model is a poor fit
plot_returnlevels(mdl, cov_f = cov_factual, cov_cf = cov_counterfactual, xlim = c(1,1000))

## 3.6 Observed probability and trend detection:
- a. Note down decision on the statistical method (see Step 3.5) in the Notes & Tables document
- b. Note dGMST, i.e. the difference between GMSTevent and GMSTpast, based on the Global Warming Index (https://www.globalwarmingindex.org) in 1 digits, e.g. 1.3 (this is the value in 2025). Check once a year that this value is still valid.
- c. Use all observational/reanalysis datasets that passed the quality checks (years see Step 3.2d). This is likely ERA5 and potentially also a station data time series.
- d. Note down/save variability or variability over mean (σ or σ/μ) and shape parameter ξ (for GEV) of the fit
- e. Note down return period in the current climate Yevent, e.g., 2025, and decide on the single rounded value that will be used for communication purposes and the model analyses – write this in the output table in Notes & Tables (bottom row). Rounding should correspond to the order of magnitude of the bandwidth, e.g., 13.876 (minimum 8.124 to maximum 20.573) should then be 14 (8-21.)
    - i. if necessary, make decision on using e.g. lower bound in case of too extreme return period. Then use this value in models analysis as well.
- f. Note down the threshold value corresponding to the event
- g. Note down probability ratio PR for dGMST, and change in intensity ΔI (absolute value for temperature, % for precipitation)

In [ ]:
%%R -i bounds
rp_factual <- return_period(mdl, x = bounds[2], fixed_cov = cov_factual)

mdl_est <- cmodel_results (mdl, rp = rp_factual, cov_f = cov_factual, cov_hist = cov_counterfactual)

mdl_est

J: we need to change this output
J: example can be found here
https://github.com/clairbarnes/wwa-current/blob/main/24-06-11_cam-heatwave/cam-heatwave_1R_trend-analysis.ipynb

check deze
https://github.com/clairbarnes/wwa-current/blob/main/00_training/25-03_mv-and-synthesis/mv.ipynb

In [ ]:
%%R
# use the built-in function to bootstrap the model results

boot_res <- boot_ci(mdl, cov_f = cov_factual, cov_cf = cov_counterfactual)
# these two lines might not even be needed
colnames(boot_res)[colnames(boot_res) == "2.5%"]  <- "lower"
colnames(boot_res)[colnames(boot_res) == "97.5%"] <- "upper"

# transpose to get the correct output format
boot_res_t <- data.frame(
    t(
        unlist(
            lapply(rownames(boot_res), function(rn) {
                setNames(
                    as.numeric(boot_res[rn, ]),
                    paste0(rn, "_", colnames(boot_res))
                )
            })
        )
    ),
    row.names = "era5"
)
boot_res_t

# save as a .csv to look at them later
# this .csv is used for the Synthesis notebook
write.csv(boot_res_t, "./data/res-obs_era5.csv")

### <font color='orange'>Select ylabels to fit your parameter</font>

In [ ]:
ylab = "Mean temperature (degC)"

In [ ]:
%%R -i ylab

# comment out the top & bottom lines to see what the figures look like before saving them
png("./data/fig_obs-trend_era5.png", height = 480, width = 480); {
    plot_trend(mdl, add_loess = T, ylim = c(x = bounds[1],x = bounds[2]), ylab = ylab, lwd = 2)
}; dev.off()

png("./data/fig_obs-gmsttrend_era5.png", height = 480, width = 480); {
    plot_covtrend(mdl, xcov = "gmst", add_loess = T, ylim = c(x = bounds[1],x = bounds[2]), ylab = ylab, lwd = 2)
}; dev.off()

png("./data/fig_obs-returnlevels_era5.png", height = 480, width = 480); {
    plot_returnlevels(mdl, cov_f = cov_factual, cov_cf = cov_counterfactual, ylim = c(x = bounds[1],x = bounds[2]), ylab = ylab,
                     legend_labels = c("2025", "Preindustrial"))
}; dev.off()

In [ ]:
%%R -i bounds
rp_factual <- return_period(mdl, x = bounds[2], fixed_cov = cov_factual)
rp_counterfactual <- return_period(mdl, x = bounds[2], fixed_cov = cov_counterfactual)

cat("Return period (factual):\n")
print(rp_factual)
cat("\nReturn period (counterfactual):\n")
print(rp_counterfactual)


In [ ]:
%%R

# try fitting different distributions to compare which actually fits better
mdl_gev <- fit_ns(dist = "gev", type = "shift", data = df, varnm = value_col, covnm = "gmst", lower = F)
mdl_norm <- fit_ns(dist = "norm", type = "shift", data = df, varnm = value_col, covnm = "gmst", lower = F)

# use AIC to identify the model that fits better (lower score is better)
cat("AIC (GEV):", aic(mdl_gev), "\n")
cat("AIC (Normal):", aic(mdl_norm), "\n")

### <font color='orange'>Select xlabel to fit your parameter</font>

In [ ]:
xlab = "Mean temperature (degC)"

In [ ]:
%%R -i xlab

# the 'rwwa' package has a built-in function to transform all points into the factual or counterfactual climate
y_factual <- stransform(mdl, fixed_cov = cov_factual)
y_counterfactual <- stransform(mdl, fixed_cov = cov_counterfactual)

# it can also extract the model parameters for a fixed value of the covariates
pars_factual <- ns_pars(mdl, fixed_cov = cov_factual)
pars_counterfactual <- ns_pars(mdl, fixed_cov = cov_counterfactual)

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# we can therefore plot the density of the temperatures as they would have been in today's climate, or in the counterfactual climate
par(lwd = 2)
plot(density(y_factual), col = "red3", ylim = c(0, 0.03), main = "", 
     xlab = xlab)

# factual model fit (dashed line)
x <- seq(15, 35, 0.1)
lines(x, devd(x, loc = pars_factual$loc, scale = pars_factual$scale, shape = pars_factual$shape),
      type = "l", col = "red3", lty = 2)

# counterfactual density and fit
lines(density(y_counterfactual), col = "blue2")
lines(x, devd(x, loc = pars_counterfactual$loc, scale = pars_counterfactual$scale, shape = pars_counterfactual$shape),
      type = "l", col = "blue2", lty = 2)

# add ticks for observed temperatures
rug(df$tmax, lwd = 3)

## 3.7 Literature review specific on the investigation of other possible climatic drivers of the event (modes of natural variability) for regions where they are important.
Steps a.i, b, c and d are essential, but if time allows it is useful to additionally do some of the optional steps.
- a. First find out what drivers influence (extremes in) the event type at the event location and its variability and trend, for example, using the following sources (see also Sec. 6a. Literature research)
    - i. Visually inspect teleconnection maps for average response of T/precip per month during years of high index https://confluence.ecmwf.int/display/COPSRV/ENSO+impacts+on+Europe and describe average impact over the region assessed if it is an ENSO year (see next step). In the maps “Global effects - temperature and precipitation” use
        - 7.a.i.1. Month of interest
        - 7.a.i.2. Variable of interest (top figures: temperature, bottom figures: precipitation)
        - 7.a.i.3. El Nino (figures left) or La Nina (figures right)
    - ii. OPTIONAL: Via IPCC AI chat https://www.climateqa.com/ e.g. by asking the question “what climate drivers influence the extremes, variability and trend of rainfall in Somalia” and, if El Niño is the answer, “what is the influence of El Niño on rainfall in Somalia?”. Any content derived from AI must be quality assured before inclusion in output.
    - iii. OPTIONAL: Search generally or in scholar.google.com for literature. ClimateQA can be used to summarise papers, in conjunction with careful proof reading of the output.
    - iv. OPTIONAL: Potentially ask this to the local experts.
- b. Assess what the current status of those drivers is on the current event. If e.g. ENSO is a driver, but ENSO is currently in a neutral state then, whilst ENSO influences the variability of the time series, it did not play a prominent role in the magnitude of the current event. Check the following and download the relevant graphics for the current status of ENSO and IOD.
    - i. ENSO
    - ii. IOD
- c. Decide whether an in-depth study should be considered and communicate this with the rest of the Team.
    - i. Investigate influence of modes of natural variability e.g. ENSO as 2nd covariate. The procedures in python notebook X can be followed.
- d. Summarise findings and decisions in scientific report Section 3.3 and summarise the findings in one sentence for the fact sheet.